In [ ]:
using Pkg
Pkg.activate("..")
using Revise

In [ ]:
using bslLD, Plots, Statistics
bslLD.greet()

# bslLD.use_cuda!()

In [ ]:
grid =  bslLD.Grid([-20.0,-4.0],[20.0,4.0],[128,129],0.01,5000,1)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)

f = bslLD.Distribution(grid, 0.001,initFuncv=initFuncv);
e = bslLD.empty_vectorfield(grid);

In [ ]:
function step(f,grid)
    bslLD.advectX!(f,grid)
    ex = @. 0.05 * sin(2pi * grid.xaxes[1] / grid.max[1])
    e = bslLD.VectorField([collect(ex)])
    bslLD.advectV!(f,grid,e)
end

function stepSelfConsitent(f,grid)
    bslLD.advectX!(f,grid)
    rho = bslLD.compute_density(f,grid)
    solution = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.AdiabaticFieldSolver())
    bslLD.advectV!(f,grid,solution.E)
end

In [ ]:
rhodiag = []
fdiag = []


for i in grid.itime
    stepSelfConsitent(f,grid)
    if i%50==0
        push!(fdiag,f.data[:,:] .- mean(f.data[:,:]))
        push!(rhodiag, bslLD.compute_density(f,grid).data[:])
    end
end


In [ ]:
num_frames = size(fdiag, 1)
frames_to_plot = 1:num_frames

animation = @animate for i in frames_to_plot
    heatmap(transpose(Array(fdiag[i])),
        title = "Frame $i",
        xlabel = "x",
        ylabel = "v",
    )

end
gif(animation, "fdiag_heatmap_animation.gif", fps = 10)

In [ ]:
heatmap(transpose(Array(fdiag[1])),
    title = "Final Frame",
    xlabel = "x",
    ylabel = "v",
)

In [ ]:
plot(Array(rhodiag[10]))


In [ ]:
plot(map(x->(mean((x.-mean(x)).^2)), Array.(rhodiag)), yscale=:log10)